# LACCI complete figures notebook

This notebook generates the paper figures and LaTeX table snippets from the outputs of:

- `minimal_results_cc18_lacci_max_v4`
- `minimal_results_cc18_lacci_mean_v4`

It uses seaborn/matplotlib for publication-style figures and the patched SHAP artifacts with explicit `row_ids`.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

RESULTS_MAX = Path("minimal_results_cc18_lacci_max_v4")
RESULTS_MEAN = Path("minimal_results_cc18_lacci_mean_v4")
FIG_DIR = RESULTS_MAX / "paper_figures_complete"
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Helpers

In [ ]:
def experiment_label(x: str) -> str:
    return {
        "tfidf_meta": "TF-IDF + meta",
        "minilm_meta": "MiniLM + meta",
        "meta_only": "Meta only",
        "none_meta": "Meta only",
    }.get(x, x)

def model_label(x: str) -> str:
    return {
        "hist_gbrt": "HGB",
        "random_forest": "RF",
    }.get(x, x)

def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


## Load consolidated tables

In [ ]:
anchor_max = read_csv_required(RESULTS_MAX / "lacci_predictive_anchor.csv")
local_max = read_csv_required(RESULTS_MAX / "lacci_local_shap_metrics_summary.csv")
anchor_mean = read_csv_required(RESULTS_MEAN / "lacci_predictive_anchor.csv")
local_mean = read_csv_required(RESULTS_MEAN / "lacci_local_shap_metrics_summary.csv")

anchor_max["Representation"] = anchor_max["experiment"].map(experiment_label)
anchor_max["Model"] = anchor_max["model"].map(model_label)
local_max["Representation"] = local_max["experiment"].map(experiment_label)
local_max["Model"] = local_max["model"].map(model_label)

anchor_mean["Representation"] = anchor_mean["experiment"].map(experiment_label)
anchor_mean["Model"] = anchor_mean["model"].map(model_label)
local_mean["Representation"] = local_mean["experiment"].map(experiment_label)
local_mean["Model"] = local_mean["model"].map(model_label)


## Figure 1 — Predictive anchor

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
plot_df = anchor_max.copy()
sns.barplot(data=plot_df, x="Representation", y="r2_mean", hue="Model", ax=ax)
ax.set_ylabel("$R^2$")
ax.set_xlabel("")
ax.set_title("Predictive anchor under maximum-performance aggregation")
plt.xticks(rotation=15)
plt.tight_layout()
fig.savefig(FIG_DIR / "figure1_predictive_anchor_max.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "figure1_predictive_anchor_max.pdf", bbox_inches="tight")
plt.show()


## Figure 2 — Global SHAP importance patterns

In [ ]:
def collect_global_top_features(results_root: Path, model_name="hist_gbrt", top_k=12):
    rows = []
    for exp in ["tfidf_meta", "minilm_meta"]:
        pkl = results_root / exp / "shap_artifacts.pkl"
        if not pkl.exists():
            continue
        artifacts = joblib.load(pkl)
        shap_folds = artifacts["task_group_kfold"][model_name]["shap_artifacts"]
        feature_names = shap_folds[0].feature_names
        mean_abs = np.mean(np.vstack([x.mean_abs_shap for x in shap_folds]), axis=0)
        idx = np.argsort(mean_abs)[::-1][:top_k]
        for j in idx:
            rows.append({
                "experiment": exp,
                "feature": feature_names[j],
                "importance": mean_abs[j],
            })
    return pd.DataFrame(rows)

global_top = collect_global_top_features(RESULTS_MAX, model_name="hist_gbrt", top_k=12)
global_top["Representation"] = global_top["experiment"].map(experiment_label)
global_top["feature_short"] = global_top["feature"].str.replace("text::", "", regex=False).str.replace("meta::", "", regex=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=False)
for ax, exp in zip(axes, ["tfidf_meta", "minilm_meta"]):
    df = global_top[global_top["experiment"] == exp].sort_values("importance")
    sns.barplot(data=df, x="importance", y="feature_short", ax=ax)
    ax.set_title(experiment_label(exp))
    ax.set_xlabel("Mean |SHAP|")
    ax.set_ylabel("")
plt.tight_layout()
fig.savefig(FIG_DIR / "figure2_global_shap_patterns.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "figure2_global_shap_patterns.pdf", bbox_inches="tight")
plt.show()


## Figure 3 — Local explanation readability and sparsity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

sns.barplot(data=local_max, x="Representation", y="readability_mean", hue="Model", ax=axes[0])
axes[0].set_title("Readability")
axes[0].set_ylabel("Mean readability")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=local_max, x="Representation", y="sparsity_mean", hue="Model", ax=axes[1])
axes[1].set_title("Sparsity")
axes[1].set_ylabel("Mean sparsity")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

for ax in axes:
    ax.legend_.remove()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIG_DIR / "figure3_local_quality.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "figure3_local_quality.pdf", bbox_inches="tight")
plt.show()


## Figures 4–5 — Case-study local SHAP explanations

In [ ]:
def topk_df(shap_fold, k=8):
    vals = np.asarray(shap_fold.shap_values[0])
    names = np.asarray(shap_fold.feature_names, dtype=object)
    idx = np.argsort(np.abs(vals))[::-1][:k]
    return pd.DataFrame({
        "feature": [str(names[i]).replace("text::", "").replace("meta::", "") for i in idx],
        "shap_value": vals[idx],
    })

tfidf_art = joblib.load(RESULTS_MAX / "tfidf_meta" / "shap_artifacts.pkl")
minilm_art = joblib.load(RESULTS_MAX / "minilm_meta" / "shap_artifacts.pkl")

tfidf_fold = tfidf_art["task_group_kfold"]["hist_gbrt"]["shap_artifacts"][0]
minilm_fold = minilm_art["task_group_kfold"]["hist_gbrt"]["shap_artifacts"][0]

# try to align on the first common row_id
common = set(tfidf_fold.row_ids.tolist()).intersection(set(minilm_fold.row_ids.tolist()))
if not common:
    raise RuntimeError("No common row_ids found between TF-IDF and MiniLM SHAP artifacts.")
chosen = sorted(common)[0]

tfidf_idx = int(np.where(tfidf_fold.row_ids == chosen)[0][0])
minilm_idx = int(np.where(minilm_fold.row_ids == chosen)[0][0])

def topk_for_index(shap_fold, idx, k=8):
    vals = np.asarray(shap_fold.shap_values[idx])
    names = np.asarray(shap_fold.feature_names, dtype=object)
    ord_idx = np.argsort(np.abs(vals))[::-1][:k]
    return pd.DataFrame({
        "feature": [str(names[i]).replace("text::", "").replace("meta::", "") for i in ord_idx],
        "shap_value": vals[ord_idx],
    })

tfidf_top = topk_for_index(tfidf_fold, tfidf_idx, k=8)
minilm_top = topk_for_index(minilm_fold, minilm_idx, k=8)

for df, title, stem in [
    (tfidf_top, "TF-IDF + meta", "figure4_case_study_tfidf"),
    (minilm_top, "MiniLM + meta", "figure5_case_study_minilm"),
]:
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    plot_df = df.iloc[::-1]
    sns.barplot(data=plot_df, x="shap_value", y="feature", ax=ax, orient="h")
    ax.set_title(title)
    ax.set_xlabel("SHAP value")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()

print("Chosen common row_id:", chosen)


## Figure 6 — Stability

In [ ]:
def topk_overlap(a, b, k=10):
    ai = np.argsort(a)[::-1][:k]
    bi = np.argsort(b)[::-1][:k]
    sa = set(ai.tolist())
    sb = set(bi.tolist())
    return len(sa & sb) / len(sa | sb) if (sa or sb) else 0.0

rows = []
for exp in ["tfidf_meta", "minilm_meta"]:
    pkl = RESULTS_MAX / exp / "shap_artifacts.pkl"
    artifacts = joblib.load(pkl)
    for model in ["hist_gbrt", "random_forest"]:
        folds = artifacts["task_group_kfold"][model]["shap_artifacts"]
        overlaps = []
        for i in range(len(folds)):
            for j in range(i + 1, len(folds)):
                overlaps.append(topk_overlap(folds[i].mean_abs_shap, folds[j].mean_abs_shap, k=10))
        rows.append({
            "Representation": experiment_label(exp),
            "Model": model_label(model),
            "mean_overlap": float(np.mean(overlaps)) if overlaps else np.nan,
        })

stab = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(6.8, 4.0))
sns.barplot(data=stab, x="Representation", y="mean_overlap", hue="Model", ax=ax)
ax.set_ylabel("Mean top-k overlap")
ax.set_xlabel("")
ax.set_title("Fold-to-fold stability of global top-k features")
plt.tight_layout()
fig.savefig(FIG_DIR / "figure6_stability.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "figure6_stability.pdf", bbox_inches="tight")
plt.show()


## LaTeX table snippets

In [ ]:
# Table I
t1 = anchor_max.copy()
t1["Representation"] = t1["Representation"]
t1["Model"] = t1["Model"]
t1["R2"] = t1.apply(lambda r: f"{r['r2_mean']:.3f} $\\pm$ {r['r2_std']:.3f}", axis=1)
t1["MAE"] = t1.apply(lambda r: f"{r['mae_mean']:.3f} $\\pm$ {r['mae_std']:.3f}", axis=1)
t1["MSE"] = t1.apply(lambda r: f"{r['mse_mean']:.3f} $\\pm$ {r['mse_std']:.3f}", axis=1)
print("TABLE I")
print(t1[["Representation", "Model", "R2", "MAE", "MSE"]].to_latex(index=False, escape=False))

# Table II
t2 = local_max.copy()
t2["Readability"] = t2.apply(lambda r: f"{r['readability_mean']:.3f} $\\pm$ {r['readability_std']:.3f}", axis=1)
t2["Sparsity"] = t2.apply(lambda r: f"{r['sparsity_mean']:.2f} $\\pm$ {r['sparsity_std']:.2f}", axis=1)
print("\nTABLE II")
print(t2[["Representation", "Model", "Readability", "Sparsity"]].to_latex(index=False, escape=False))

# Table III
mean_merge = anchor_mean.merge(local_mean, on=["experiment", "model", "Representation", "Model"], how="inner")
mean_merge["R2"] = mean_merge.apply(lambda r: f"{r['r2_mean']:.3f} $\\pm$ {r['r2_std']:.3f}", axis=1)
mean_merge["Readability"] = mean_merge.apply(lambda r: f"{r['readability_mean']:.3f} $\\pm$ {r['readability_std']:.3f}", axis=1)
mean_merge["Sparsity"] = mean_merge.apply(lambda r: f"{r['sparsity_mean']:.2f} $\\pm$ {r['sparsity_std']:.2f}", axis=1)
print("\nTABLE III")
print(mean_merge[["Representation", "Model", "R2", "Readability", "Sparsity"]].to_latex(index=False, escape=False))


## Final notes

This notebook explicitly aligns the case-study figures using `row_ids` from the patched codebase, rather than relying on sampled position within a fold.